# ***TLC Trip Record Data***

## Installamos las dependencias para poder descargar los datos usando wget

In [3]:
!pip install wget

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9686 sha256=57cba9bb20b94480e48b0ab5729584445c721d344c90e915c20788ea0e537e5c
  Stored in directory: /home/omar/.cache/pip/wheels/01/46/3b/e29ffbe4ebe614ff224bad40fc6a5773a67a163251585a13a9
Successfully built wget


In [6]:
pip install pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 11.5 MB/s  0:00:04m0:00:0100:01
Note: you may need to restart the kernel to use updated packages.


## En este ejemplo solo voy a usar los datos de un solo mes en enero del 2026 y especificamente de los taxis amarillos

In [41]:
import wget
url = 'https://d37ci6vzurychx.cloudfront.net/trip-dTata/yellow_tripdata_2026-01.parquet'
file_output = wget.download(url) 

100% [........................................................................] 64165080 / 64165080

In [37]:
file_output= 'yellow_tripdata_2026-01.parquet'

## Empezamos a leer la informacion en formato parquet usando pyarrow para poder transformar los datos a una tabla que podemos modificar

In [38]:
import pyarrow.parquet as pq

yellowTripData = pq.read_table(file_output)

# **Hadoop**
## Voy a usar hadoop para almacenar todos los datos que descarguemos y tambien los datos procesados, limpios pero primero hay que configurar la conexion al servidor local de hadoop

In [39]:
import os
import subprocess

os.environ["HADOOP_HOME"] = "/home/omar/opt/hadoop"

os.environ["JAVA_HOME"] = subprocess.check_output(
    "dirname $(dirname $(readlink -f $(which java)))",
    shell=True,
    text=True
).strip()

os.environ["CLASSPATH"] = subprocess.check_output(
    ["hadoop", "classpath"],
    text=True
).strip()

print("HADOOP_HOME =", os.environ["HADOOP_HOME"])
print("JAVA_HOME   =", os.environ["JAVA_HOME"])
print("CLASSPATH   =", os.environ["CLASSPATH"][:200], "...")

HADOOP_HOME = /home/omar/opt/hadoop
JAVA_HOME   = /usr/lib/jvm/java-17-openjdk-amd64
CLASSPATH   = /home/omar/opt/hadoop/etc/hadoop:/home/omar/opt/hadoop/share/hadoop/common/lib/*:/home/omar/opt/hadoop/share/hadoop/common/*:/home/omar/opt/hadoop/share/hadoop/hdfs:/home/omar/opt/hadoop/share/hadoop/ ...


## Se crea el directorio donde se va a almacenar la informacion para que despues pyspark pueda leerla desde el servidor de hadoop

In [40]:
import pyarrow.fs as fs

local = fs.LocalFileSystem()

hdfs = fs.HadoopFileSystem(
    host="localhost",
    port=9000
)

2026-09-18 17:11:54,013 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [41]:
hdfs.create_dir("/tlc/raw/yellow", recursive=True)

In [42]:
hdfs_file = "/tlc/raw/yellow/yellow_tripdata_2026-01.parquet"

with open(file_output, "rb") as src:
        with hdfs.open_output_stream(hdfs_file) as dst:
            while chunk := src.read(1024 * 1024):
                dst.write(chunk)


In [43]:
info = hdfs.get_file_info(
    "/tlc/raw/yellow/yellow_tripdata_2026-01.parquet"
)

print(info)

<FileInfo for '/tlc/raw/yellow/yellow_tripdata_2026-01.parquet': type=FileType.File, size=64165080>


# **Spark**
## Creamos la session de spark con la que vamos a realizar el analisis de los datos y leemos los datos desde el servidor de Hadoop donde vamos a tener toda nuestra informacion almacenada

In [44]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("NYCTaxiAnalysis")
    .config(
        "spark.jars.packages",
        "com.mysql:mysql-connector-j:9.4.0"
    )
    .getOrCreate()
)

df = spark.read.parquet(
    "hdfs://localhost:9000/tlc/raw/yellow/yellow_tripdata_2026-01.parquet"
)

df.printSchema()
#df.show(10)

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



## Contamos la cantidad de datos que tenemos

In [45]:
print("Number of rows:", df.count())

df.describe()

Number of rows: 3724889


DataFrame[summary: string, VendorID: string, passenger_count: string, trip_distance: string, RatecodeID: string, store_and_fwd_flag: string, PULocationID: string, DOLocationID: string, payment_type: string, fare_amount: string, extra: string, mta_tax: string, tip_amount: string, tolls_amount: string, improvement_surcharge: string, total_amount: string, congestion_surcharge: string, Airport_fee: string, cbd_congestion_fee: string]

## Contamos los valores null dentro de las columnas

In [46]:
from pyspark.sql.functions import col, sum
null_counts = df.select(
    [
        sum(col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ]
)
null_counts.select("passenger_count","RatecodeID","store_and_fwd_flag","Airport_fee","congestion_surcharge").show()

+---------------+----------+------------------+-----------+--------------------+
|passenger_count|RatecodeID|store_and_fwd_flag|Airport_fee|congestion_surcharge|
+---------------+----------+------------------+-----------+--------------------+
|        1088058|   1088058|           1088058|    1088058|             1088058|
+---------------+----------+------------------+-----------+--------------------+



## Podemos ver que tenemos al menos un millon de datos con valores nulos en algunas columnas, la mas importante seria passenger_count ya que nos indica la cantidad de pasajeros que habia durante el viaje

## Obtenemos informacion importante de los la distancia, cuanto pagaron en total y el numero de pasajeros

In [47]:
df.select(
    "trip_distance",
    "fare_amount",
    "payment_type",
    "total_amount",
    "passenger_count"
).describe().show()

+-------+-----------------+------------------+------------------+------------------+------------------+
|summary|    trip_distance|       fare_amount|      payment_type|      total_amount|   passenger_count|
+-------+-----------------+------------------+------------------+------------------+------------------+
|  count|          3724889|           3724889|           3724889|           3724889|           2636831|
|   mean|6.455646860885151|20.804253893199448|0.8465637499533543|29.178525515798285| 1.256271258946819|
| stddev|648.8855284529166|18.927007021274658|0.7120492979865746|22.585529763602636|0.6702431378098668|
|    min|              0.0|           -2555.2|                 0|           -2560.2|                 0|
|    max|        269097.48|            2555.2|                 4|            2560.2|                 9|
+-------+-----------------+------------------+------------------+------------------+------------------+



In [48]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    hour,
    avg,
    sum,
    count,
    unix_timestamp,
    date_format,
    dayofweek,
    sum
)

## Limpiamos un poco los datos por ejemplo eliminamos los datos negativos o menores a cero en la distancia, el numero de pasajeros y el pago total

In [49]:
clean_df = (
    df
    .filter(col("trip_distance") > 0)
    .filter(col("total_amount") > 0)
    .filter(col("passenger_count") > 0)
)
#clean_df.show()

## Duracion promedio de los viajes en minutos

In [96]:
time_min_df = df.withColumn(
    "trip_duration_minutes",
    (
        unix_timestamp("tpep_dropoff_datetime")
        - unix_timestamp("tpep_pickup_datetime")
    ) / 60
)
time_min_df = time_min_df.filter(
    (col("trip_duration_minutes") > 0) &
    (col("trip_duration_minutes") < 600)
)

time_min_df =time_min_df.select("trip_duration_minutes").describe()
time_min_df.show()

+-------+---------------------+
|summary|trip_duration_minutes|
+-------+---------------------+
|  count|              3678797|
|   mean|   17.088416856560897|
| stddev|   14.572335015985303|
|    min| 0.016666666666666666|
|    max|                599.6|
+-------+---------------------+



## Informacion acerca general acerca de la hora en la toman los taxis, el dia de la semana y la fecha

In [97]:
pickup_info = (
    clean_df
    .withColumn(
        "pickup_hour",
        hour("tpep_pickup_datetime")
    )
    .withColumn(
        "pickup_day_of_week",
        dayofweek("tpep_pickup_datetime")
    )
    .withColumn(
        "pickup_date",
        date_format(
            "tpep_pickup_datetime",
            "yyyy-MM-dd"
        )
    )
)

pickup_info = pickup_info.select("pickup_hour","pickup_day_of_week","pickup_date").describe()
pickup_info.show()

[Stage 147:================================================>      (14 + 2) / 16]

+-------+------------------+------------------+-----------+
|summary|       pickup_hour|pickup_day_of_week|pickup_date|
+-------+------------------+------------------+-----------+
|  count|           2552307|           2552307|    2552307|
|   mean|14.359021857480311|  4.42049330272573|       NULL|
| stddev| 5.524580320058522|1.8977893758720095|       NULL|
|    min|                 0|                 1| 2025-12-31|
|    max|                23|                 7| 2026-02-01|
+-------+------------------+------------------+-----------+



## El numero de viajes por hora

In [52]:
trips_by_hour = (
    pickup_info
    .groupBy("pickup_hour")
    .count()
    .orderBy("pickup_hour")
)

trips_by_hour.show()

+-----------+------+
|pickup_hour| count|
+-----------+------+
|          0| 59319|
|          1| 38369|
|          2| 26138|
|          3| 17783|
|          4| 12516|
|          5| 16363|
|          6| 33398|
|          7| 66487|
|          8| 93219|
|          9|111006|
|         10|122154|
|         11|131534|
|         12|144977|
|         13|150534|
|         14|161257|
|         15|172529|
|         16|175247|
|         17|184380|
|         18|185249|
|         19|158653|
+-----------+------+
only showing top 20 rows


## El precio por hora

In [53]:
fare_by_hour = (
    pickup_info
    .groupBy("pickup_hour")
    .agg(
        avg("fare_amount").alias("average_fare"),
        avg("trip_distance").alias("average_distance")
    )
    .orderBy("pickup_hour")
)

fare_by_hour.show()

+-----------+------------------+------------------+
|pickup_hour|      average_fare|  average_distance|
+-----------+------------------+------------------+
|          0| 20.59955494866733| 4.073139129115455|
|          1|  17.9136771873127| 3.387060126664774|
|          2| 16.72318922641365|3.0607016604177777|
|          3|18.005851656076022|3.4258398470449243|
|          4|23.126310322786857|  4.88110258868648|
|          5| 29.08172767829855| 6.739510480963151|
|          6|25.127116593807962| 5.571965087729803|
|          7|20.695856784032916| 4.088460601320563|
|          8| 19.08143543698171|3.4233338697046842|
|          9|18.849382465812717| 4.149617227897585|
|         10|18.856751313915144|3.4887567333038536|
|         11|18.579245518269094|3.0473295117612125|
|         12|18.984478779392774| 3.135348296626372|
|         13|19.285360051549738|3.1885164813264777|
|         14|20.292980273724513| 3.417770639414084|
|         15|20.435667974659246| 3.440708170800283|
|         16

## Las ganancias por dia

In [54]:
revenue_by_day = (
    pickup_info
    .groupBy("pickup_date")
    .agg(
        sum("total_amount").alias("total_revenue"),
        count("*").alias("total_trips")
    )
    .orderBy("pickup_date")
)

print("Number of rows:", revenue_by_day.count())
revenue_by_day.show()

Number of rows: 33
+-----------+------------------+-----------+
|pickup_date|     total_revenue|total_trips|
+-----------+------------------+-----------+
| 2025-12-31|329.84999999999997|          5|
| 2026-01-01|1951752.2799999474|      64013|
| 2026-01-02|2225444.1399999172|      72931|
| 2026-01-03| 2260596.399999944|      76821|
| 2026-01-04|2047451.7099999427|      64546|
| 2026-01-05|2289868.2699999306|      74130|
| 2026-01-06|2400790.4500000062|      82860|
| 2026-01-07| 2468466.719999961|      86579|
| 2026-01-08|2660787.9399999375|      91686|
| 2026-01-09|  2637128.61999997|      92117|
| 2026-01-10| 2543510.979999948|      95633|
| 2026-01-11| 2250941.459999944|      77993|
| 2026-01-12| 2461796.689999965|      84845|
| 2026-01-13|2653173.6499999664|      92662|
| 2026-01-14|  2789452.10999998|      98076|
| 2026-01-15|2971809.2399999755|     104356|
| 2026-01-16| 2721701.720000002|      94910|
| 2026-01-17| 2157087.449999933|      82792|
| 2026-01-18|1952136.5799999547|    

## Los metodos de pagos mas utilizados

In [55]:
payment_analysis = (
    clean_df
    .groupBy("payment_type")
    .agg(
        count("*").alias("total_trips"),
        avg("fare_amount").alias("average_fare"),
        avg("tip_amount").alias("average_tip")
    )
    .orderBy("payment_type")
)

payment_analysis.show()

+------------+-----------+------------------+--------------------+
|payment_type|total_trips|      average_fare|         average_tip|
+------------+-----------+------------------+--------------------+
|           1|    2219235|19.378425912533544|   4.113503707358173|
|           2|     296124| 19.44152216638991|4.145222947143764E-4|
|           3|       9218| 19.88712410501194|1.833369494467346...|
|           4|      27730|23.721828705373323|0.002022719076812117|
+------------+-----------+------------------+--------------------+



## El lugar mas comun donde toman el taxi

In [56]:
pickup_analysis = (
    clean_df
    .groupBy("PULocationID")
    .count()
    .orderBy("count", ascending=False)
)
pickup_analysis.show()

+------------+------+
|PULocationID| count|
+------------+------+
|         132|142317|
|         237|138082|
|         236|125656|
|         161|120020|
|         186| 95845|
|         162| 92349|
|         142| 88818|
|         230| 84276|
|         138| 81634|
|         239| 72390|
|         163| 71272|
|         170| 68202|
|         234| 67771|
|         141| 65654|
|          68| 65494|
|          48| 59128|
|         140| 56299|
|          79| 54972|
|         263| 51569|
|         238| 50654|
+------------+------+
only showing top 20 rows


## La columna con el ID no nos dice mucho por si solo asi que tenemos que incluir otro archivo donde tenemos mas informacion acerca de estos id

In [57]:
from pyarrow import csv

taxi_zone_file = "taxi_zone_lookup.csv"
hdfs_file = "/tlc/raw/yellow/taxi_zone_lookup.csv"

with open(taxi_zone_file, "rb") as src:
        with hdfs.open_output_stream(hdfs_file) as dst:
            while chunk := src.read(1024 * 1024):
                dst.write(chunk)


## Despues de subir este archivo a hadoop lo leemos desde pyspark

In [58]:
df_zn = spark.read.csv("hdfs://localhost:9000/tlc/raw/yellow/taxi_zone_lookup.csv")
df_zn= df_zn.withColumnRenamed("_c0","LocationID").withColumnRenamed("_c1","Borough").withColumnRenamed("_c2","Zone").withColumnRenamed("_c3","Service_zone")
df_zn =df_zn.filter(df_zn.LocationID !="LocationID")


## Con esto podemos analizar el numero de viajes por zona usando el LocationID

In [59]:
pickup_zones = (
    clean_df
    .groupBy("PULocationID")
    .agg(count("*").alias("total_pickups"))
    .join(
        df_zn,
        clean_df.PULocationID == df_zn.LocationID
    )
    .select(
        "LocationID",
        "Borough",
        "Zone",
        "total_pickups"
    )
    .orderBy("total_pickups", ascending=False)
)

pickup_zones.show(20, truncate=False)

+----------+---------+----------------------------+-------------+
|LocationID|Borough  |Zone                        |total_pickups|
+----------+---------+----------------------------+-------------+
|132       |Queens   |JFK Airport                 |142317       |
|237       |Manhattan|Upper East Side South       |138082       |
|236       |Manhattan|Upper East Side North       |125656       |
|161       |Manhattan|Midtown Center              |120020       |
|186       |Manhattan|Penn Station/Madison Sq West|95845        |
|162       |Manhattan|Midtown East                |92349        |
|142       |Manhattan|Lincoln Square East         |88818        |
|230       |Manhattan|Times Sq/Theatre District   |84276        |
|138       |Queens   |LaGuardia Airport           |81634        |
|239       |Manhattan|Upper West Side South       |72390        |
|163       |Manhattan|Midtown North               |71272        |
|170       |Manhattan|Murray Hill                 |68202        |
|234      

## Destinos con la mayor cantidad de viajes

In [60]:
dropoff_zones = (
    clean_df
    .groupBy("DOLocationID")
    .agg(count("*").alias("total_dropoffs"))
    .join(
        df_zn,
        clean_df.DOLocationID == df_zn.LocationID
    )
    .select(
        "LocationID",
        "Borough",
        "Zone",
        "total_dropoffs"
    )
    .orderBy("total_dropoffs", ascending=False)
)

dropoff_zones.show(20, truncate=False)

+----------+---------+-----------------------------+--------------+
|LocationID|Borough  |Zone                         |total_dropoffs|
+----------+---------+-----------------------------+--------------+
|236       |Manhattan|Upper East Side North        |131656        |
|237       |Manhattan|Upper East Side South        |122465        |
|161       |Manhattan|Midtown Center               |93801         |
|239       |Manhattan|Upper West Side South        |77225         |
|142       |Manhattan|Lincoln Square East          |76891         |
|141       |Manhattan|Lenox Hill West              |75581         |
|230       |Manhattan|Times Sq/Theatre District    |74035         |
|170       |Manhattan|Murray Hill                  |71303         |
|162       |Manhattan|Midtown East                 |68341         |
|238       |Manhattan|Upper West Side North        |62701         |
|68        |Manhattan|East Chelsea                 |62508         |
|163       |Manhattan|Midtown North             

## Numero de viajes por Borough

In [61]:
pickup_boroughs = (
    clean_df
    .join(
        df_zn,
        clean_df.PULocationID == df_zn.LocationID
    )
    .groupBy("Borough")
    .agg(
        count("*").alias("total_pickups")
    )
    .orderBy("total_pickups", ascending=False)
)

pickup_boroughs.show()

+-------------+-------------+
|      Borough|total_pickups|
+-------------+-------------+
|    Manhattan|      2222810|
|       Queens|       259336|
|     Brooklyn|        47093|
|        Bronx|        18343|
|      Unknown|         3843|
|          N/A|          674|
|          EWR|          117|
|Staten Island|           91|
+-------------+-------------+



## Este analisis para ver los viajes mas comunes de cierta zona a otra

In [62]:
pickup_zones_df = df_zn.alias("pickup")
dropoff_zones_df = df_zn.alias("dropoff")

routes = (
    clean_df
    .join(
        df_zn,
        (clean_df.DOLocationID == df_zn.LocationID) #& (clean_df.DOLocationID == df_zn.LocationID)
    )
    .groupBy(
        clean_df.PULocationID.alias("pickup_zone"),
        clean_df.DOLocationID.alias("dropoff_zone"),
        df_zn.Zone
    )
    .agg(
        count("*").alias("total_trips")
    )
    .orderBy("total_trips", ascending=False)
)

routes.show(20, truncate=False)

[Stage 96:=============================================>          (13 + 3) / 16]

+-----------+------------+-------------------------+-----------+
|pickup_zone|dropoff_zone|Zone                     |total_trips|
+-----------+------------+-------------------------+-----------+
|237        |236         |Upper East Side North    |22583      |
|236        |237         |Upper East Side South    |18966      |
|236        |236         |Upper East Side North    |15541      |
|237        |237         |Upper East Side South    |14459      |
|161        |237         |Upper East Side South    |9602       |
|237        |161         |Midtown Center           |8572       |
|142        |239         |Upper West Side South    |8275       |
|239        |238         |Upper West Side North    |7929       |
|161        |236         |Upper East Side North    |7792       |
|141        |236         |Upper East Side North    |7580       |
|239        |142         |Lincoln Square East      |7574       |
|132        |265         |Outside of NYC           |7125       |
|236        |141         

## Precios promedios por zona

In [63]:
fare_by_zone = (
    clean_df
    .join(
        df_zn,
        clean_df.PULocationID == df_zn.LocationID
    )
    .groupBy(
        "Borough",
        "Zone"
    )
    .agg(
        count("*").alias("total_trips"),
        avg("fare_amount").alias("average_fare"),
        avg("total_amount").alias("average_total")
    )
    .orderBy("average_total", ascending=False)
)

fare_by_zone = fare_by_zone.filter(
    (col("total_trips") > 10)
)
fare_by_zone.show(20, truncate=False)

+-------------+--------------------------------+-----------+------------------+------------------+
|Borough      |Zone                            |total_trips|average_fare      |average_total     |
+-------------+--------------------------------+-----------+------------------+------------------+
|EWR          |Newark Airport                  |117        |95.51880341880342 |111.2045299145299 |
|N/A          |Outside of NYC                  |674        |76.47816023738875 |88.36629080118698 |
|Queens       |Flushing Meadows-Corona Park    |312        |69.87189102564102 |86.27759615384615 |
|Queens       |South Ozone Park                |1626       |68.79029520295202 |80.08563345633443 |
|Queens       |JFK Airport                     |142317     |61.5403582144086  |79.28013301291337 |
|Queens       |Saint Michaels Cemetery/Woodside|131        |62.59007633587786 |78.49129770992367 |
|Queens       |Baisley Park                    |1175       |62.935080851063816|75.72347234042542 |
|Queens   

## Distancia promedio por zona

In [64]:
distance_by_zone = (
    clean_df
    .join(
        df_zn,
        clean_df.PULocationID == df_zn.LocationID
    )
    .groupBy(
        "Borough",
        "Zone"
    )
    .agg(
        count("*").alias("total_trips"),
        avg("trip_distance").alias("average_distance")
    )
    .orderBy("average_distance", ascending=False)
)

distance_by_zone.show(20, truncate=False)

+-------------+-----------------------------------+-----------+------------------+
|Borough      |Zone                               |total_trips|average_distance  |
+-------------+-----------------------------------+-----------+------------------+
|Queens       |Old Astoria                        |614        |165.9911889250812 |
|Staten Island|Eltingville/Annadale/Prince's Bay  |1          |30.19             |
|Staten Island|Great Kills                        |2          |21.24             |
|Staten Island|Stapleton                          |5          |18.048000000000002|
|Queens       |Breezy Point/Fort Tilden/Riis Beach|3          |17.09             |
|Queens       |Far Rockaway                       |787        |16.781029224904703|
|Staten Island|Saint George/New Brighton          |6          |15.883333333333333|
|Staten Island|New Dorp/Midland Beach             |3          |15.536666666666667|
|Queens       |Rosedale                           |317        |15.397728706624608|
|Que

## Ganancias totales por zona

In [65]:
revenue_by_zone = (
    clean_df
    .join(
        df_zn,
        clean_df.PULocationID == df_zn.LocationID
    )
    .groupBy(
        "Borough",
        "Zone"
    )
    .agg(
        count("*").alias("total_trips"),
        sum("total_amount").alias("total_revenue")
    )
    .orderBy("total_revenue", ascending=False)
)

revenue_by_zone.show(20, truncate=False)

+---------+----------------------------+-----------+--------------------+
|Borough  |Zone                        |total_trips|total_revenue       |
+---------+----------------------------+-----------+--------------------+
|Queens   |JFK Airport                 |142317     |1.1282910689998792E7|
|Queens   |LaGuardia Airport           |81634      |5662571.37999998    |
|Manhattan|Midtown Center              |120020     |3018611.879999988   |
|Manhattan|Upper East Side South       |138082     |2827869.069999995   |
|Manhattan|Upper East Side North       |125656     |2629032.019999981   |
|Manhattan|Penn Station/Madison Sq West|95845      |2391736.8000000035  |
|Manhattan|Times Sq/Theatre District   |84276      |2355177.9100000225  |
|Manhattan|Midtown East                |92349      |2262026.040000017   |
|Manhattan|Lincoln Square East         |88818      |1954589.7300000025  |
|Manhattan|Midtown North               |71272      |1745125.919999992   |
|Manhattan|East Chelsea               

## Numero de viajes por zona y por hora del dia

In [67]:
zone_hour = (
    pickup_info
    .join(
        df_zn,
        pickup_info.PULocationID == df_zn.LocationID
    )
    .groupBy(
        "Zone",
        "Borough",
        "pickup_hour"
    )
    .agg(
        count("*").alias("total_trips")
    )
    .orderBy(
        "Zone",
        "pickup_hour",
         ascending=False
    )
)
zone_hour.show(20, truncate=False)

+--------------+---------+-----------+-----------+
|Zone          |Borough  |pickup_hour|total_trips|
+--------------+---------+-----------+-----------+
|Yorkville West|Manhattan|23         |1343       |
|Yorkville West|Manhattan|22         |2161       |
|Yorkville West|Manhattan|21         |2594       |
|Yorkville West|Manhattan|20         |2756       |
|Yorkville West|Manhattan|19         |3126       |
|Yorkville West|Manhattan|18         |3696       |
|Yorkville West|Manhattan|17         |3442       |
|Yorkville West|Manhattan|16         |3338       |
|Yorkville West|Manhattan|15         |3310       |
|Yorkville West|Manhattan|14         |3181       |
|Yorkville West|Manhattan|13         |3084       |
|Yorkville West|Manhattan|12         |3003       |
|Yorkville West|Manhattan|11         |2952       |
|Yorkville West|Manhattan|10         |2932       |
|Yorkville West|Manhattan|9          |2503       |
|Yorkville West|Manhattan|8          |2185       |
|Yorkville West|Manhattan|7    

## Ganancias promedio por zona y por hora

In [68]:
zone_hour_revenue = (
    pickup_info
    .join(
        df_zn,
        clean_df.PULocationID == df_zn.LocationID
    )
    .groupBy(
        "Borough",
        "Zone",
        "pickup_hour"
    )
    .agg(
        count("*").alias("total_trips"),
        sum("total_amount").alias("total_revenue"),
        avg("total_amount").alias("average_fare")
    )
    .orderBy(
        "total_revenue",
        "average_fare",
         ascending=False
    )
)
zone_hour_revenue.show(20, truncate=False)

+-------+-----------------+-----------+-----------+------------------+-----------------+
|Borough|Zone             |pickup_hour|total_trips|total_revenue     |average_fare     |
+-------+-----------------+-----------+-----------+------------------+-----------------+
|Queens |JFK Airport      |16         |10683      |955659.5900000171 |89.45610689881279|
|Queens |JFK Airport      |20         |11229      |882370.8800000092 |78.57964912280784|
|Queens |JFK Airport      |15         |9741       |827464.5700000082 |84.94657324710073|
|Queens |JFK Airport      |19         |9992       |827108.4800000031 |82.77706965572489|
|Queens |JFK Airport      |17         |8541       |756518.6899999997 |88.57495492331105|
|Queens |JFK Airport      |21         |9932       |743636.2500000019 |74.87275976641179|
|Queens |JFK Airport      |22         |9753       |735390.2500000033 |75.40144058238525|
|Queens |JFK Airport      |18         |8451       |718708.6800000017 |85.04421725239636|
|Queens |JFK Airport 

# **MySQL**
## Por ultimo guardamos toda esta informacion dentro de nuestra base de datos para despues mostrarla desde streamlit

In [69]:
jdbc_url = "jdbc:mysql://localhost:3306/taxi_db"

properties = {
    "user": "spark",
    "password": "spark",
    "driver": "com.mysql.cj.jdbc.Driver"
}

In [70]:
spark.conf.get("spark.jars.packages")

'com.mysql:mysql-connector-j:9.4.0'

In [39]:
#test_df = spark.read.jdbc(
#    url=jdbc_url,
#    table="test_table",
#    properties=properties
#)
#
#test_df.show()

+---+
| ID|
+---+
+---+



In [98]:
time_min_df.write.jdbc(
    url=jdbc_url,
    table="Viajes_duracion",
    mode="overwrite",
    properties=properties
)

In [99]:
pickup_info.write.jdbc(
    url=jdbc_url,
    table="Viajes_horarios",
    mode="overwrite",
    properties=properties
)

In [100]:

trips_by_hour.write.jdbc(
    url=jdbc_url,
    table="Viajes_horas",
    mode="overwrite",
    properties=properties
)

In [101]:

fare_by_hour.write.jdbc(
    url=jdbc_url,
    table="Tarifa_hora",
    mode="overwrite",
    properties=properties
)

In [102]:
revenue_by_day.write.jdbc(
    url=jdbc_url,
    table="Ganancia_dia",
    mode="overwrite",
    properties=properties
)

In [103]:
payment_analysis.write.jdbc(
    url=jdbc_url,
    table="Metodos_pago",
    mode="overwrite",
    properties=properties
)

In [104]:
pickup_analysis.write.jdbc(
    url=jdbc_url,
    table="Viajes_analisis",
    mode="overwrite",
    properties=properties
)

In [105]:
pickup_zones.write.jdbc(
    url=jdbc_url,
    table="Viajes_zona",
    mode="overwrite",
    properties=properties
)

In [106]:
dropoff_zones.write.jdbc(
    url=jdbc_url,
    table="Destinos_zona",
    mode="overwrite",
    properties=properties
)

In [107]:
pickup_boroughs.write.jdbc(
    url=jdbc_url,
    table="Viajes_distrito",
    mode="overwrite",
    properties=properties
)

In [108]:
routes.write.jdbc(
    url=jdbc_url,
    table="Rutas",
    mode="overwrite",
    properties=properties
)

In [109]:
fare_by_zone.write.jdbc(
    url=jdbc_url,
    table="Tarifa_zona",
    mode="overwrite",
    properties=properties
)

In [110]:
distance_by_zone.write.jdbc(
    url=jdbc_url,
    table="Distancia_zona",
    mode="overwrite",
    properties=properties
)

In [111]:
revenue_by_zone.write.jdbc(
    url=jdbc_url,
    table="Ganancia_zona",
    mode="overwrite",
    properties=properties
)

In [112]:

zone_hour.write.jdbc(
    url=jdbc_url,
    table="Viajes_zona_hora",
    mode="overwrite",
    properties=properties
)

In [113]:
zone_hour_revenue.write.jdbc(
    url=jdbc_url,
    table="Vaijes_zona_ganancia",
    mode="overwrite",
    properties=properties
)